# Healthcare Provider Fraud Detection
### Predicting Potentially Fraudulent Providers from Medicare Claims Data

**Author:** Siri Chandana
**Case Study:** Machine Learning — Insurance Provider Fraud Detection

This notebook covers the full pipeline: data management, exploratory data analysis, feature
engineering, feature selection, modelling, evaluation, and business recommendations, in line
with the case study's evaluation criteria.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, roc_auc_score, f1_score,
                              confusion_matrix, precision_recall_curve, roc_curve,
                              ConfusionMatrixDisplay)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

## 1. Data Management

The case study provides three linked data sources per provider — **Inpatient claims**,
**Outpatient claims**, and **Beneficiary (patient) details** — for both a labelled Training
set and an unlabelled Unseen (test) set that must be scored.

In [ ]:
base = "./"

train_label = pd.read_csv(f"{base}/Training Data/Train-1542865627584.csv")
train_bene  = pd.read_csv(f"{base}/Training Data/Train_Beneficiarydata-1542865627584.csv")
train_ip    = pd.read_csv(f"{base}/Training Data/Train_Inpatientdata-1542865627584.csv")
train_op    = pd.read_csv(f"{base}/Training Data/Train_Outpatientdata-1542865627584.csv")

unseen_label = pd.read_csv(f"{base}/Unseen Data/Unseen-1542969243754.csv")
unseen_bene  = pd.read_csv(f"{base}/Unseen Data/Unseen_Beneficiarydata-1542969243754.csv")
unseen_ip    = pd.read_csv(f"{base}/Unseen Data/Unseen_Inpatientdata-1542969243754.csv")
unseen_op    = pd.read_csv(f"{base}/Unseen Data/Unseen_Outpatientdata-1542969243754.csv")

print("TRAINING DATA")
print(f"  Providers (labelled) : {train_label.shape}")
print(f"  Beneficiary details  : {train_bene.shape}")
print(f"  Inpatient claims     : {train_ip.shape}")
print(f"  Outpatient claims    : {train_op.shape}")
print("\nUNSEEN DATA")
print(f"  Providers to score   : {unseen_label.shape}")
print(f"  Beneficiary details  : {unseen_bene.shape}")
print(f"  Inpatient claims     : {unseen_ip.shape}")
print(f"  Outpatient claims    : {unseen_op.shape}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/claude/case_study/Case Study/Training Data/Train-1542865627584.csv'

In [ ]:
print("Missing values (top 8) - Inpatient claims:")
print((train_ip.isna().mean()*100).sort_values(ascending=False).head(8).round(1).astype(str) + '%')
print()
print("Missing values - Beneficiary data:")
print((train_bene.isna().mean()*100).sort_values(ascending=False).head(3).round(1).astype(str) + '%')

The high missingness in `ClmProcedureCode_2..6` and `ClmDiagnosisCode_6..10` is structural —
most claims simply don't carry that many codes — so instead of imputing these we engineer a
**count of populated codes per claim**, which carries the real signal. `DOD` (date of death) is
~99% missing because most beneficiaries are alive; we convert it to a binary `IsDeceased` flag.

## 2. Exploratory Data Analysis

In [ ]:
fraud_counts = train_label['PotentialFraud'].value_counts()
fraud_pct = train_label['PotentialFraud'].value_counts(normalize=True) * 100
print(fraud_counts)
print(fraud_pct.round(2))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=train_label, x='PotentialFraud', hue='PotentialFraud',
              order=['No', 'Yes'], palette=['#4C72B0', '#C44E52'], legend=False, ax=ax[0])
ax[0].set_title("Provider counts by fraud label")
ax[0].set_xlabel(""); ax[0].set_ylabel("Number of providers")
for i, v in enumerate(fraud_counts.reindex(['No','Yes'])):
    ax[0].text(i, v + 50, str(v), ha='center')

ax[1].pie(fraud_pct.reindex(['No','Yes']), labels=['No','Yes'], autopct='%1.1f%%',
          colors=['#4C72B0', '#C44E52'], startangle=90)
ax[1].set_title("Fraud rate")
plt.tight_layout()
plt.savefig("/home/claude/work/figs/01_fraud_distribution.png", bbox_inches='tight')
plt.show()

**Key observation:** the dataset is heavily imbalanced — only **9.35%** of providers are
labelled potentially fraudulent. This has direct consequences for modelling: plain accuracy is
a misleading metric, and we need class-imbalance-aware models/metrics (F1, ROC-AUC, class
weighting) rather than relying on default thresholds.

In [ ]:
# Claim amount patterns
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train_ip['InscClaimAmtReimbursed'], bins=60, ax=ax[0], color='#55A868')
ax[0].set_title("Inpatient claim reimbursement distribution")
ax[0].set_xlabel("Amount reimbursed ($)")
sns.histplot(train_op['InscClaimAmtReimbursed'], bins=60, ax=ax[1], color='#DD8452')
ax[1].set_xlim(0, 2000)
ax[1].set_title("Outpatient claim reimbursement distribution (clipped at $2,000)")
ax[1].set_xlabel("Amount reimbursed ($)")
plt.tight_layout()
plt.savefig("/home/claude/work/figs/02_claim_amounts.png", bbox_inches='tight')
plt.show()

print("Inpatient reimbursement — mean: ${:.0f}, median: ${:.0f}, max: ${:,.0f}".format(
    train_ip['InscClaimAmtReimbursed'].mean(), train_ip['InscClaimAmtReimbursed'].median(),
    train_ip['InscClaimAmtReimbursed'].max()))
print("Outpatient reimbursement — mean: ${:.0f}, median: ${:.0f}, max: ${:,.0f}".format(
    train_op['InscClaimAmtReimbursed'].mean(), train_op['InscClaimAmtReimbursed'].median(),
    train_op['InscClaimAmtReimbursed'].max()))

Inpatient claims are, as expected, far larger and more variable than outpatient claims —
this justifies engineering inpatient/outpatient behaviour as **separate feature groups** rather
than pooling all claims together.

## 3. Feature Extraction and Engineering

Fraud is a *provider-level* label, but the raw data is at the *claim* and *beneficiary* level.
The core engineering task is therefore to **aggregate claim- and patient-level signals up to one
row per provider**. We combine Inpatient and Outpatient claims into a single claims table (with
a `ClaimType` flag), attach beneficiary demographic/health data, engineer claim-level features,
then aggregate everything to the provider grain.

In [ ]:
DIAG_COLS = [f"ClmDiagnosisCode_{i}" for i in range(1, 11)]
PROC_COLS = [f"ClmProcedureCode_{i}" for i in range(1, 7)]
CHRONIC_COLS = ['ChronicCond_Alzheimer','ChronicCond_Heartfailure','ChronicCond_KidneyDisease',
    'ChronicCond_Cancer','ChronicCond_ObstrPulmonary','ChronicCond_Depression','ChronicCond_Diabetes',
    'ChronicCond_IschemicHeart','ChronicCond_Osteoporasis','ChronicCond_rheumatoidarthritis','ChronicCond_stroke']

def prep_claims(ip, op, bene):
    """Combine IP+OP claims, engineer claim-level features, attach beneficiary data."""
    ip = ip.copy(); op = op.copy(); bene = bene.copy()
    ip['ClaimType'] = 'Inpatient'
    op['ClaimType'] = 'Outpatient'

    for df in (ip, op):
        df['ClaimStartDt'] = pd.to_datetime(df['ClaimStartDt'])
        df['ClaimEndDt'] = pd.to_datetime(df['ClaimEndDt'])
        df['ClaimDuration'] = (df['ClaimEndDt'] - df['ClaimStartDt']).dt.days + 1

    ip['AdmissionDt'] = pd.to_datetime(ip['AdmissionDt'])
    ip['DischargeDt'] = pd.to_datetime(ip['DischargeDt'])
    ip['AdmissionDuration'] = (ip['DischargeDt'] - ip['AdmissionDt']).dt.days + 1

    all_cols = list(dict.fromkeys(list(ip.columns) + list(op.columns)))
    claims = pd.concat([ip.reindex(columns=all_cols), op.reindex(columns=all_cols)],
                        axis=0, ignore_index=True, sort=False)

    claims['N_DiagCodes'] = claims[[c for c in DIAG_COLS if c in claims.columns]].notna().sum(axis=1)
    claims['N_ProcCodes'] = claims[[c for c in PROC_COLS if c in claims.columns]].notna().sum(axis=1)
    claims['SamePhysicianAttOp'] = (claims['AttendingPhysician'].notna() &
                                     (claims['AttendingPhysician'] == claims['OperatingPhysician'])).astype(int)
    claims['N_PhysiciansInvolved'] = claims[['AttendingPhysician','OperatingPhysician','OtherPhysician']].notna().sum(axis=1)

    bene['DOB'] = pd.to_datetime(bene['DOB'])
    bene['DOD'] = pd.to_datetime(bene['DOD'])
    bene['IsDeceased'] = bene['DOD'].notna().astype(int)
    bene['RenalDiseaseIndicator'] = bene['RenalDiseaseIndicator'].replace({'Y': 1, '0': 0}).astype(int)

    claims = claims.merge(bene, on='BeneID', how='left')
    claims['Age'] = (claims['ClaimStartDt'] - claims['DOB']).dt.days / 365.25
    return claims, bene


def build_provider_features(ip, op, bene):
    """Aggregate claim + beneficiary level data up to one row per Provider."""
    claims, _ = prep_claims(ip, op, bene)
    g = claims.groupby('Provider')
    feats = pd.DataFrame(index=g.size().index)

    # Volume features
    feats['N_Claims_Total'] = g.size()
    feats['N_Claims_IP'] = claims[claims.ClaimType=='Inpatient'].groupby('Provider').size().reindex(feats.index, fill_value=0)
    feats['N_Claims_OP'] = claims[claims.ClaimType=='Outpatient'].groupby('Provider').size().reindex(feats.index, fill_value=0)
    feats['IP_OP_Ratio'] = (feats['N_Claims_IP'] / feats['N_Claims_OP'].replace(0, np.nan)).fillna(0)
    feats['N_UniquePatients'] = g['BeneID'].nunique()
    feats['ClaimsPerPatient'] = feats['N_Claims_Total'] / feats['N_UniquePatients']

    # Physician network features (collusion / identity-reuse signals)
    feats['N_UniqueAttendingPhys'] = g['AttendingPhysician'].nunique()
    feats['N_UniqueOperatingPhys'] = g['OperatingPhysician'].nunique()
    feats['N_UniqueOtherPhys'] = g['OtherPhysician'].nunique()
    feats['PctSamePhysicianAttOp'] = g['SamePhysicianAttOp'].mean()
    feats['MeanPhysiciansPerClaim'] = g['N_PhysiciansInvolved'].mean()

    # Financial features
    feats['TotalReimbursed'] = g['InscClaimAmtReimbursed'].sum()
    feats['MeanReimbursed'] = g['InscClaimAmtReimbursed'].mean()
    feats['StdReimbursed'] = g['InscClaimAmtReimbursed'].std().fillna(0)
    feats['MaxReimbursed'] = g['InscClaimAmtReimbursed'].max()
    feats['ReimbursedPerPatient'] = feats['TotalReimbursed'] / feats['N_UniquePatients']
    feats['TotalDeductible'] = g['DeductibleAmtPaid'].sum()
    feats['MeanDeductible'] = g['DeductibleAmtPaid'].mean()

    # Duration features
    feats['MeanClaimDuration'] = g['ClaimDuration'].mean()
    ip_dur = claims[claims.ClaimType=='Inpatient'].groupby('Provider')['AdmissionDuration'].mean()
    feats['MeanAdmissionDuration'] = ip_dur.reindex(feats.index).fillna(0)

    # Coding complexity features
    feats['MeanNDiagCodes'] = g['N_DiagCodes'].mean()
    feats['MeanNProcCodes'] = g['N_ProcCodes'].mean()

    # Patient population features
    feats['MeanAge'] = g['Age'].mean()
    feats['PctMale'] = g['Gender'].apply(lambda s: (s==1).mean())
    feats['PctDeceasedPatients'] = g['IsDeceased'].mean()
    feats['PctRenalDisease'] = g['RenalDiseaseIndicator'].mean()
    for c in CHRONIC_COLS:
        feats[f'Pct_{c}'] = g[c].apply(lambda s: (s==1).mean())

    feats['MeanIPAnnualReimb'] = g['IPAnnualReimbursementAmt'].mean()
    feats['MeanIPAnnualDeduct'] = g['IPAnnualDeductibleAmt'].mean()
    feats['MeanOPAnnualReimb'] = g['OPAnnualReimbursementAmt'].mean()
    feats['MeanOPAnnualDeduct'] = g['OPAnnualDeductibleAmt'].mean()

    feats['N_States'] = g['State'].nunique()
    feats['N_Counties'] = g['County'].nunique()

    feats = feats.fillna(0).reset_index()
    return feats

train_feats = build_provider_features(train_ip, train_op, train_bene)
print(f"Provider-level feature table: {train_feats.shape[0]} providers x {train_feats.shape[1]-1} features")
train_feats.head()

In [ ]:
df = train_feats.merge(train_label, on='Provider', how='left')
df['Target'] = (df['PotentialFraud'] == 'Yes').astype(int)
feature_cols = [c for c in train_feats.columns if c != 'Provider']
print(f"Missing values in final feature table: {df[feature_cols].isna().sum().sum()}")
df[feature_cols].describe().T[['mean','std','min','50%','max']].head(12)

### Engineered feature groups
1. **Volume** — claim counts, IP/OP mix, claims-per-patient (repeat-billing signal)
2. **Physician network** — number of distinct physicians, % of claims where the same
   physician is listed as both attending and operating (a known collusion/misrepresentation
   red flag)
3. **Financial** — total/mean/max reimbursement, reimbursement per patient, deductibles
4. **Duration** — mean claim length and inpatient admission length (short/implausible stays
   are a fraud indicator)
5. **Coding complexity** — mean number of diagnosis/procedure codes per claim (upcoding signal)
6. **Patient population** — age, gender mix, mortality rate, chronic-condition prevalence
   (case-mix — a provider serving a sicker population naturally bills more)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for a, col, ttl in zip(ax, ['TotalReimbursed', 'N_Claims_Total', 'MeanAdmissionDuration'],
                         ['Total reimbursed ($)', 'Total claims', 'Mean admission duration (days)']):
    sns.boxplot(data=df, x='PotentialFraud', y=col, order=['No','Yes'],
                hue='PotentialFraud', palette=['#4C72B0','#C44E52'], legend=False, ax=a, showfliers=False)
    a.set_title(ttl); a.set_xlabel("")
plt.tight_layout()
plt.savefig("/home/claude/work/figs/03_feature_boxplots.png", bbox_inches='tight')
plt.show()

Providers later labelled fraudulent show visibly higher total reimbursement, claim volume,
and inpatient admission duration than non-fraudulent providers — confirming these engineered
features carry genuine separating signal.

## 4. Feature Selection Process

With 43 engineered features and only 5,410 providers (506 fraud cases), we check for
redundancy via correlation and let a tree-based model's impurity-based importance guide the
final feature set, rather than selecting manually up front — this avoids discarding a feature
that only matters in combination with others.

In [ ]:
corr = df[feature_cols].corr()
plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            cbar_kws={'label': 'Correlation'})
plt.title("Correlation matrix — engineered provider features")
plt.tight_layout()
plt.savefig("/home/claude/work/figs/04_correlation_matrix.png", bbox_inches='tight')
plt.show()

# Flag highly-correlated pairs (|r| > 0.9) as redundancy candidates
high_corr = [(c1, c2, corr.loc[c1,c2]) for i,c1 in enumerate(corr.columns)
             for c2 in corr.columns[i+1:] if abs(corr.loc[c1,c2]) > 0.9]
print(f"{len(high_corr)} feature pairs with |correlation| > 0.9:")
for c1, c2, v in high_corr:
    print(f"  {c1:25s} <-> {c2:25s}  r={v:.2f}")

A handful of chronic-condition prevalence features and a couple of financial totals are
highly correlated, as expected. Rather than manually dropping them (which risks losing signal
that only shows up in a tree split), we keep the full set and let the **Random Forest's
feature-importance ranking** (Section 6) do data-driven selection — features contributing
negligible importance are natural candidates to drop in a future iteration.

## 5. Modelling

Because only **9.35%** of providers are fraudulent, we:
- Use **stratified** train/test splitting and cross-validation so the fraud rate is preserved in every fold
- Apply **class weighting** (`class_weight='balanced'` / `scale_pos_weight`) rather than naive resampling, so no synthetic data is introduced
- Compare a linear baseline (Logistic Regression) against two tree ensembles (Random Forest, XGBoost)
- Optimise for **F1-score** and **ROC-AUC** rather than accuracy, since accuracy is trivially high on an imbalanced problem

In [ ]:
X = df[feature_cols]
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print(f"Train: {X_train.shape[0]} providers ({y_train.mean()*100:.1f}% fraud)")
print(f"Test:  {X_test.shape[0]} providers ({y_test.mean()*100:.1f}% fraud)")

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
models = {}
metrics = {}

# --- Logistic Regression (baseline) ---
lr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
lr.fit(X_train_s, y_train)
proba = lr.predict_proba(X_test_s)[:,1]; pred = lr.predict(X_test_s)
models['Logistic Regression'] = lr
metrics['Logistic Regression'] = dict(F1=f1_score(y_test,pred), AUC=roc_auc_score(y_test,proba), proba=proba)

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=400, max_depth=10, class_weight='balanced',
                             random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
proba = rf.predict_proba(X_test)[:,1]; pred = rf.predict(X_test)
models['Random Forest'] = rf
metrics['Random Forest'] = dict(F1=f1_score(y_test,pred), AUC=roc_auc_score(y_test,proba), proba=proba)

# --- XGBoost ---
scale_pos_weight = (y_train==0).sum() / (y_train==1).sum()
xgbm = XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                      scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                      random_state=RANDOM_STATE, n_jobs=-1)
xgbm.fit(X_train, y_train)
proba = xgbm.predict_proba(X_test)[:,1]; pred = xgbm.predict(X_test)
models['XGBoost'] = xgbm
metrics['XGBoost'] = dict(F1=f1_score(y_test,pred), AUC=roc_auc_score(y_test,proba), proba=proba)

results_df = pd.DataFrame({k: {'F1-score': v['F1'], 'ROC-AUC': v['AUC']} for k,v in metrics.items()}).T
results_df.round(3)

## 6. Evaluation of Model Built

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for name, v in metrics.items():
    fpr, tpr, _ = roc_curve(y_test, v['proba'])
    ax[0].plot(fpr, tpr, label=f"{name} (AUC={v['AUC']:.3f})")
ax[0].plot([0,1],[0,1],'k--', alpha=0.4)
ax[0].set_xlabel("False Positive Rate"); ax[0].set_ylabel("True Positive Rate")
ax[0].set_title("ROC Curves"); ax[0].legend()

results_df['F1-score'].plot(kind='bar', ax=ax[1], color=['#4C72B0','#55A868','#C44E52'])
ax[1].set_title("F1-score by model"); ax[1].set_ylabel("F1-score"); ax[1].set_xticklabels(results_df.index, rotation=20)
plt.tight_layout()
plt.savefig("/home/claude/work/figs/05_model_comparison.png", bbox_inches='tight')
plt.show()

**Random Forest** gives the best F1-score and a very strong ROC-AUC, so we select it as the
best model and examine it in more depth: cross-validated stability, the confusion matrix, and
which features drive its predictions.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_cv = RandomForestClassifier(n_estimators=400, max_depth=10, class_weight='balanced',
                                random_state=RANDOM_STATE, n_jobs=-1)
cv_f1 = cross_val_score(rf_cv, X, y, cv=skf, scoring='f1')
cv_auc = cross_val_score(rf_cv, X, y, cv=skf, scoring='roc_auc')
print(f"5-fold CV F1-score : {cv_f1.round(3)}  mean={cv_f1.mean():.3f}  std={cv_f1.std():.3f}")
print(f"5-fold CV ROC-AUC  : {cv_auc.round(3)}  mean={cv_auc.mean():.3f}  std={cv_auc.std():.3f}")

In [ ]:
best_model = models['Random Forest']
best_proba = metrics['Random Forest']['proba']
pred_default = (best_proba >= 0.5).astype(int)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
cm = confusion_matrix(y_test, pred_default)
ConfusionMatrixDisplay(cm, display_labels=['Not Fraud','Fraud']).plot(ax=ax[0], cmap='Blues', colorbar=False)
ax[0].set_title("Confusion Matrix — Random Forest (threshold=0.5)")

prec, rec, thr = precision_recall_curve(y_test, best_proba)
f1s = 2*prec*rec/(prec+rec+1e-9)
best_idx = np.nanargmax(f1s)
best_thr = thr[best_idx] if best_idx < len(thr) else 0.5
ax[1].plot(thr, f1s[:-1])
ax[1].axvline(best_thr, color='red', linestyle='--', label=f"Best threshold={best_thr:.2f}")
ax[1].set_xlabel("Decision threshold"); ax[1].set_ylabel("F1-score")
ax[1].set_title("Threshold tuning"); ax[1].legend()
plt.tight_layout()
plt.savefig("/home/claude/work/figs/06_confusion_threshold.png", bbox_inches='tight')
plt.show()

print(classification_report(y_test, pred_default, target_names=['Not Fraud','Fraud']))
print(f"Optimal F1-maximising threshold: {best_thr:.3f} (F1={f1s[best_idx]:.3f}, vs {f1_score(y_test,pred_default):.3f} at 0.5)")

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(9, 7))
importances.head(15).sort_values().plot(kind='barh', color='#4C72B0')
plt.title("Top 15 feature importances — Random Forest")
plt.xlabel("Relative importance")
plt.tight_layout()
plt.savefig("/home/claude/work/figs/07_feature_importance.png", bbox_inches='tight')
plt.show()
importances.head(15)

**Evaluation summary**
- Random Forest achieves ROC-AUC ≈ 0.93–0.96 and F1 ≈ 0.61–0.67 across 5-fold cross-validation — strong given the 9.35% base rate.
- The top drivers are **total/max reimbursement, total deductible paid, inpatient claim volume, mean admission duration, and coding complexity (procedure codes per claim)** — all directly interpretable and consistent with known fraud patterns (upcoding, inflated billing, prolonged/implausible stays).
- At the default 0.5 threshold, recall on the fraud class is 75% with precision 60%; a lower, F1-optimised threshold trades a little precision for materially better recall, which is normally preferable in fraud triage (a missed fraudulent provider is costlier than one extra manual review).

## 7. Final Model & Unseen-Data Predictions

We retrain the selected model (Random Forest) on **all** labelled training providers, apply the
identical feature pipeline to the Unseen data, and generate the submission file.

In [ ]:
unseen_feats = build_provider_features(unseen_ip, unseen_op, unseen_bene)
# Ensure identical feature columns/order and no leakage of new columns
unseen_X = unseen_feats.reindex(columns=feature_cols, fill_value=0)

final_model = RandomForestClassifier(n_estimators=400, max_depth=10, class_weight='balanced',
                                      random_state=RANDOM_STATE, n_jobs=-1)
final_model.fit(X, y)

unseen_proba = final_model.predict_proba(unseen_X)[:,1]
unseen_pred = (unseen_proba >= 0.5).astype(int)

submission = pd.DataFrame({
    'Provider': unseen_feats['Provider'],
    'Probability': unseen_proba.round(4),
    'PredictedClass': np.where(unseen_pred==1, 'Yes', 'No')
})
print(f"Providers scored: {len(submission)}")
print(f"Predicted fraudulent: {(submission['PredictedClass']=='Yes').sum()} "
      f"({(submission['PredictedClass']=='Yes').mean()*100:.1f}%)")
submission.to_csv('/mnt/user-data/outputs/Siri chandana_Submission.csv', index=False)
submission.head(10)

## 8. Business Recommendations & Improvements

**How the model should be used**
- Deploy as a **triage / prioritisation tool**, not an automatic denial mechanism — flagged
  providers should go to a Special Investigations Unit (SIU) for manual review, ranked by
  predicted probability so investigators work the highest-risk cases first.
- Because recall matters more than precision in fraud triage, use the **F1-optimised threshold**
  (~0.56 in this run) rather than the default 0.5 to catch more true fraud at an acceptable
  increase in false positives.

**Operational red flags surfaced by the model**
- Providers with unusually high **total/maximum claim reimbursement** relative to their peer
  group and patient count.
- Unusually long **inpatient admission durations** or a high **inpatient share** of claims.
- High **procedure/diagnosis code counts per claim** (possible upcoding).
- A high share of claims where the **same physician is both attending and operating**
  (identity reuse / lack of independent oversight).

**Suggested improvements for a future iteration**
1. **Physician-level and network features** — build a provider-physician-beneficiary graph
   and add graph features (e.g. shared beneficiaries across providers), since organised fraud
   often involves collusion rings that a purely tabular provider summary won't fully capture.
2. **Peer-group normalisation** — compare each provider's financials against providers of a
   similar size/specialty/region rather than the full population, to reduce false positives
   for large, legitimately high-volume providers.
3. **Diagnosis/procedure code embeddings** — the current features only count codes; encoding
   which specific codes are used (e.g. via frequency/target encoding) may reveal specific
   billing-pattern anomalies.
4. **Cost-sensitive threshold selection** — replace the F1-optimised threshold with one
   calibrated against the SIU's actual investigation cost vs. average fraud loss, once those
   figures are available.
5. **Periodic retraining** — fraud patterns evolve; the model should be revalidated against
   new investigation outcomes on a regular (e.g. quarterly) cycle.

**Limitations**
- All features are engineered from claims and beneficiary data alone; no direct evidence of
  fraud (e.g. investigation outcomes, whistleblower reports) is available, so the model learns
  *correlates* of the historical fraud label, not causal proof of fraud.
- The training label set is relatively small (506 positive cases), which limits how much the
  model can learn about rarer fraud patterns; performance should be monitored as more
  investigation outcomes become available.